# Détection de coups grâce à la pose

### Chargement du modèle de pose et des résultats

In [ ]:
from ultralytics import YOLO
model = YOLO('roboflow/models/yolo11x-pose.pt')       
video_path = '/Users/philomenecarrel/4A/projet_info_ping_pong/PingPongTracking/roboflow/dataset_labelise/Sombre_echange/sombre_echanges - Trim.mp4'   # Votre vidéo de match
results = model(video_path, vid_stride=10)  


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/120) /Users/philomenecarrel/4A/projet_info_ping_pong/PingPongTracking/roboflow/dataset_labelise/Sombre_echange/sombre_echanges - Trim.mp4: 480x640 1 person, 436.2ms
video 1/1 (frame 2/120) /Users/philomenecarrel/4A/projet_info_ping_pong/PingPongTracking/roboflow/dataset_labelise/Sombre_echange/sombre_echanges - Trim.mp4: 480x640 1 person, 389.0ms
video 1/1 (frame 3/120) /Users/philomenecarrel/4A/projet_info_ping_pong/PingPongTracking/r

KeyboardInterrupt: 

### Logique de classification des coups


In [ ]:
def classify_stroke(pose_results, dominant_hand="right"):
    # Get keypoints (x, y)
    if pose_results[0].keypoints is not None:
        kp = pose_results[0].keypoints.xy[0]
        shoulder_l, shoulder_r = kp[5], kp[6]
        wrist_idx = 10 if dominant_hand == "right" else 9
        wrist_x = kp[wrist_idx][0]
        shoulder_x = shoulder_r[0] if dominant_hand == "right" else shoulder_l[0]
        if dominant_hand == "right":
            if wrist_x > shoulder_x:
                return "Forehand"
            else:
                return "Backhand"
        else: 
            if wrist_x < shoulder_x:
                return "Forehand"
            else:
                return "Backhand"

            

In [12]:
classify_stroke(results, dominant_hand="right")

TypeError: 'generator' object is not subscriptable

In [ ]:
def boxes_intersect(box_ball, box_racket):
    # box format: [x1, y1, x2, y2]
    return not (box_ball[2] < box_racket[0] or # Balle à gauche de raquette
                box_ball[0] > box_racket[2] or # Balle à droite de raquette
                box_ball[3] < box_racket[1] or # Balle au-dessus de raquette
                box_ball[1] > box_racket[3])   # Balle en dessous de raquette

### Test de détéection des coups et impacts

In [ ]:
import cv2
from ultralytics import YOLO

model_obj = YOLO('models/best-2.pt')
model_pose = YOLO('models/yolo11x-pose.pt')

cap = cv2.VideoCapture("dataset_labelise/video_simple/video_simple.mp4")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # Détection de Balle et Raquette
    results_obj = model_obj(frame,conf=0.6, iou=0.5, verbose=False)[0]
    ball_box = None
    racket_box = None
    for box in results_obj.boxes:
        cls = int(box.cls[0])
        if cls == 0: ball_box = box.xyxy[0].cpu().numpy()   # Classe Balle
        if cls == 1: racket_box = box.xyxy[0].cpu().numpy() # Classe Raquette

    # Vérification de la superposition
    if ball_box is not None and racket_box is not None:
        if boxes_intersect(ball_box, racket_box):
            
            # On ne demande la POSE que si ça se touche 
            results_pose = model_pose(frame, verbose=False)[0]
            
            if results_pose.keypoints:
                # Analyse de la pose pour classer le coup
                stroke = classify_stroke(results_pose)
                print(f"IMPACT DÉTECTÉ : {stroke}")
                cv2.putText(frame, stroke, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("Arbitrage", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

IMPACT DÉTECTÉ : Backhand
IMPACT DÉTECTÉ : Backhand


### Code final pour affichage de coups détectés directement sur la vidéo

In [ ]:
import cv2
import time
from ultralytics import YOLO

model_obj = YOLO('models/best-2.pt') 
model_pose = YOLO('models/yolo11x-pose.pt')

DUREE_AFFICHAGE = 1.5 
affichage_expiration = 0
dernier_coup = ""

def boxes_intersect(box_ball, box_racket):
    return not (box_ball[2] < box_racket[0] or 
                box_ball[0] > box_racket[2] or 
                box_ball[3] < box_racket[1] or 
                box_ball[1] > box_racket[3])

def classifier_coup(keypoints):
    kp = keypoints.xy[0].cpu().numpy()
    if len(kp) < 11: return "Inconnu"
    return "COUP DROIT" if kp[10][0] < kp[8][0] else "REVERS"


cap = cv2.VideoCapture("dataset_labelise/video_simple/video_simple.mp4")
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    temps_actuel = time.time()
    results_obj = model_obj(frame, verbose=False, conf=0.3)[0]

    ball_box_for_logic = None
    racket_box_for_logic = None

    for box in results_obj.boxes:
        cls = int(box.cls[0])
        coords = box.xyxy[0].cpu().numpy().astype(int) 
        if cls == 0: # la balle
            ball_box_for_logic = coords 
            cv2.rectangle(frame, (coords[0], coords[1]), (coords[2], coords[3]), (255, 0, 0), 2)
            cv2.putText(frame, "Balle", (coords[0], coords[1]-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
            
        elif cls == 1: # la raquette
            racket_box_for_logic = coords 
            cv2.rectangle(frame, (coords[0], coords[1]), (coords[2], coords[3]), (0, 0, 255), 2)
            cv2.putText(frame, "Raquette", (coords[0], coords[1]-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

    if ball_box_for_logic is not None and racket_box_for_logic is not None:
        if boxes_intersect(ball_box_for_logic, racket_box_for_logic):
            # L'impact a lieu : on lance la pose
            results_pose = model_pose(frame, verbose=False)[0]
            
            if results_pose.keypoints:
                dernier_coup = classifier_coup(results_pose.keypoints)
                # On planifie l'affichage du texte pour plus tard
                affichage_expiration = temps_actuel + DUREE_AFFICHAGE

    # AFFICHAGE PERSISTANT DU TEXTE (Si impact récent)
    if temps_actuel < affichage_expiration:
        cv2.putText(frame, f"IMPACT : {dernier_coup}", (50, 80), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

    cv2.imshow("Arbitrage Automatique", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()